## Second step

In `second_step.ipynb`, we will use the records collected in `first_step.ipynb` to perform our first filtering step: filtering by the country where the specimen was recorded. This step is necessary to focus on a specific study region.

We will adopt a practice of creating a new database field whenever we perform a filter, storing the filtered records in this new field to preserve the original ones. By convention, the name of this field will always be `original-field_upd`. In this case, the name will be `country_upd`, since the original field name is `country`.

If you do not wish to perform this filtering, continue using the `country` field instead of `country_upd` for subsequent filtering steps.

In [ ]:
from library import *

specieslink, db_config = configure()

In [ ]:
conn = mysql_conn.connect(**db_config)
cursor = conn.cursor()

coluna = "country"
table = "biodiversity_records"

sql = f"""SELECT {coluna} FROM {table} WHERE {coluna} IS NOT NULL GROUP BY {coluna}"""

cursor.execute(sql)
results = cursor.fetchall()

for result in results:
    print(result[0])

cursor.close()
conn.close()  

Identify the names that correspond to the country you need. For example, if searcing for Brazil possible variations of its writing could be "Brazil, Brasil, Brésil, BR..." and those would be the ones we would use.

Note that you must enter the values ​​without spaces between them, so, Brasil,Brazil,Brésil...

In [ ]:
def filtering(field_input, update_input, filters_input, table):
    if not field_input:
        print("please provide a field")
        return
    
    conn = mysql_conn.connect(**db_config)
    cursor = conn.cursor()

    try:
        sql = f"ALTER TABLE {table} ADD COLUMN {field_input} TEXT"

        cursor.execute(sql)
        conn.commit()

        print(f"field '{field_input}' created with succes on table '{table}'")

        cursor.close()
        conn.close()  
    except Exception as e:
        print(f"field '{field_input}' exists already or error while creating: {e}")
        
    filters = {}
    if '=' not in filters_input:
        print("badly formatted filter: use field=value1,value2,...")
        return

    key, value = filters_input.split('=', 1)
    values = [v.strip() for v in value.split(',') if v.strip()]
    filters[key.strip()] = values

    update_values = {}
    for item in update_input.split():
        if '=' not in item: #
            print(f"badly formatted update value: {item} - use key=value")
            return
        else:
            key, value = item.split('=', 1)
            update_values[key.strip()] = value.strip()

    filter_field, filter_values = next(iter(filters.items()))
    update_field, update_value = next(iter(update_values.items()))
    for value in filter_values:
        specieslink.update_records(filters={filter_field: value}, update_values={update_field: update_value}, db_config=db_config, table=table)

In [ ]:
field_input = input("specify the name you want the new field to have (do not create the field manually when running via the pipeline!)").strip()
update_input = input("specify the field and the new value to update (key=value format, separated by a space): ").strip()
filters_input = input("specify the field and the old value to be updated (key=value format, separated by spaces): ").strip()
table = "biodiversity_records"
print(f"executing specieslink.update_records(filters={filters_input}, update_values={update_input}, table={table})...\n")

filtering(field_input=field_input, update_input=update_input, filters_input=filters_input, table=table)

In [ ]:
conn = mysql_conn.connect(**db_config)
cursor = conn.cursor()
table = "biodiversity_records"

sql = f"""
SELECT
    COUNT(*) AS total_records,
    SUM(country_att IS NOT NULL) AS used_records
FROM {table}
"""

cursor.execute(sql)
total_records, used_records = cursor.fetchone()

cursor.close()
conn.close()

In [ ]:
plt.figure(figsize=(6, 4))

x = [0]

plt.bar(
    x,
    [total_records],
    alpha=0.5,
    width=0.15,
    label='total'
)

plt.bar(
    x,
    [used_records],
    width=0.15,
    label='with country_upd'
)

plt.xlim(-0.2, 0.3)
plt.ylabel('total of records')
plt.title('sample of records to be used')
plt.legend()

plt.text(0, total_records, str(total_records), ha='center', va='bottom')
plt.text(0, used_records, str(used_records), ha='center', va='bottom')

plt.tight_layout()
plt.show()